# 15. 推理系统与部署优化体系

训练一个模型只是开始。真实落地时更常见的瓶颈是：

- 显存不够
- 延迟太高
- 吞吐太低
- 长上下文成本太高

因此，理解推理系统优化并不是“部署细节”，而是现代大模型能力的一部分。

## 推理系统与部署优化的形式化定义

推理系统优化是指围绕已训练好的模型，在显存、吞吐、时延和并发约束下，对执行路径、缓存布局、数值表示和调度策略进行系统级重构的过程。它研究的不是“模型会不会答”，而是“模型能否在现实资源边界内高效地答”。

## 输入、输出与参数化方式

输入是请求流、上下文序列和模型权重；系统状态包括 KV Cache、批处理队列、量化权重、调度器状态与缓存分页结构；输出是生成 token 流和相应的资源消耗特征。

## 结构分解与信息流

            现代大模型推理栈通常包含以下关键子系统：

- 权重加载与量化执行层
- 注意力 kernel 与 KV Cache 管理层
- 请求调度与连续批处理层
- 解码优化层，如 speculative decoding

因此推理系统本质上是“模型计算图 + 缓存管理 + 调度器”的复合架构。

## 优化目标与训练机制

            推理优化的目标不是单一指标最大化，而是在多个系统指标间做折中：

- 降低首 token 时延
- 提高 tokens/s 吞吐
- 提高显存利用率
- 控制精度损失

这也是为什么量化、PagedAttention、Continuous Batching 和 Speculative Decoding 会被同时讨论，因为它们分别对应不同瓶颈。

## 计算复杂度、统计性质与工程代价

            对于长上下文大模型，注意力与 KV Cache 成本通常是主瓶颈。随着上下文增长，显存占用会迅速上升；随着并发上升，调度效率成为吞吐上限的关键。

因此，系统优化常常带来比“单纯换更大 GPU”更高的边际收益。

## 与相邻模型的差异

            与训练优化相比，推理优化更关注在线服务约束；与单模型架构改造相比，推理优化更强调系统资源利用率。
在实际生产中，模型架构、量化策略与调度系统往往必须协同设计，而不能彼此割裂。

## 先建立直觉

            很多人以为“大模型能力”只由参数和数据决定。
但在真实产品里，用户感受到的往往是：

- 回得快不快
- 长上下文撑不撑得住
- 成本能不能接受
- 并发高时会不会崩

这些问题大多属于推理系统，而不是模型公式本身。

## 架构是怎么工作的

            推理系统可以理解成围绕模型主干搭起来的一整套基础设施：

- KV Cache 管理
- 请求调度
- 批处理与连续批处理
- 量化执行
- 注意力 kernel 优化
- speculative decoding

所以同一个模型，在不同推理系统里，成本和速度可能差很多。

## 训练时到底发生了什么

            这一节虽然不训练大模型，但它和训练并不脱节。

因为很多架构设计之所以出现，就是为了更好的推理：

- GQA / MQA 减少 KV cache
- MoE 控制激活计算量
- 更轻量的量化格式降低显存

也就是说，现代模型架构越来越不是“只为训练而设计”，而是训练和推理一起考虑。

## 什么时候该用它

            如果你未来想做线上部署，这节几乎是必修：

- 显存紧张时优先考虑量化
- 长上下文时优先考虑 KV cache 设计
- 高并发服务时优先考虑 continuous batching
- 低时延场景时优先考虑 speculative decoding

## 最常见的误区

            常见误区：

1. **误以为换更大 GPU 就能解决所有问题**
   系统调度和 cache 管理常常同样关键。

2. **误以为量化只会带来精度损失**
   在合适方法下，精度损失可能很有限，但收益非常大。

3. **误以为推理优化只是工程问题**
   它会反过来影响模型架构选择和产品策略。

## 1. KV Cache 显存公式

对于 Decoder-only 模型，KV Cache 大致随下面因素线性增长：

$$
\mathrm{Memory} \propto 2 \times L \times T \times H_{kv} \times d_{head} \times \mathrm{bytes}
$$

其中：

- $L$：层数
- $T$：上下文长度
- $H_{kv}$：KV heads 数
- $d_{head}$：每个 head 的维度
- `2`：因为要存 K 和 V

这也是为什么：

- GQA / MQA 很重要
- 长上下文成本很高
- KV Cache 管理决定了推理系统上限

## 2. 量化

一个常见的均匀量化形式是：

$$
q = \mathrm{round}\left(\frac{x}{s}\right) + z
$$

反量化：

$$
\hat{x} = s(q-z)
$$

量化的目标是：

- 降低显存
- 提高吞吐
- 尽量少损失精度

常见路线包括：

- INT8
- INT4
- GPTQ
- AWQ
- FP8

## 3. PagedAttention、Continuous Batching、Speculative Decoding

### PagedAttention

核心思想：像操作系统分页那样管理 KV Cache，减少碎片，提高显存利用率。

### Continuous Batching

不必等整批请求全部完成再加入新请求，而是持续把新请求插入 scheduler，提高 GPU 利用率。

### Speculative Decoding

用一个更小更快的 draft model 先提议若干 token，再由大模型并行验证，从而降低单 token 延迟。

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]



print("临时目录:", temp_root)

In [ ]:
# 计算不同 KV head 设计下的 cache 显存。
context_lengths = np.array([4_096, 8_192, 16_384, 32_768, 65_536, 131_072])
num_layers = 32
head_dim = 128
bytes_per_value = 2  # fp16


def kv_cache_gb(num_kv_heads):
    memory_bytes = 2 * num_layers * context_lengths * num_kv_heads * head_dim * bytes_per_value
    return memory_bytes / (1024 ** 3)


kv_df = pd.DataFrame(
    {
        "上下文长度": context_lengths,
        "MQA (1 KV head)": kv_cache_gb(1),
        "GQA (8 KV heads)": kv_cache_gb(8),
        "MHA (32 KV heads)": kv_cache_gb(32),
    }
)
kv_df

In [ ]:
kv_long = kv_df.melt(id_vars="上下文长度", var_name="设计", value_name="KV Cache 显存(GB)")
plt.figure(figsize=(11, 6))
sns.lineplot(data=kv_long, x="上下文长度", y="KV Cache 显存(GB)", hue="设计", marker="o")
plt.xscale("log", base=2)
plt.title("不同 KV 设计下，KV Cache 显存随上下文长度增长")
plt.show()

In [ ]:
# 量化误差实验：比较不同 bit-width 的重建误差。
np.random.seed(42)
weights = np.random.randn(20000).astype(np.float32)


def uniform_quantize(x, bits):
    qmin = -(2 ** (bits - 1))
    qmax = 2 ** (bits - 1) - 1
    scale = np.max(np.abs(x)) / qmax
    q = np.clip(np.round(x / scale), qmin, qmax)
    x_hat = q * scale
    return x_hat


quant_records = []
for bits in [16, 8, 6, 4]:
    x_hat = uniform_quantize(weights, bits=bits)
    mse = np.mean((weights - x_hat) ** 2)
    quant_records.append({"bits": bits, "重建 MSE": mse})

quant_df = pd.DataFrame(quant_records)
quant_df

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=quant_df, x="bits", y="重建 MSE", palette="magma")
plt.title("不同量化位宽的重建误差")
plt.show()

In [ ]:
# Toy speculative decoding：不同 acceptance rate 下的理论 speedup。
acceptance_rates = np.linspace(0.1, 0.95, 18)
draft_lengths = [2, 4, 8]

spec_records = []
for k in draft_lengths:
    for acc in acceptance_rates:
        # 一个简单近似：每轮平均接受 1 + (k-1)*acc 个 token
        speedup = 1 + (k - 1) * acc
        spec_records.append({"draft 长度": k, "接受率": acc, "理论加速比": speedup})

spec_df = pd.DataFrame(spec_records)

In [ ]:
plt.figure(figsize=(11, 6))
sns.lineplot(data=spec_df, x="接受率", y="理论加速比", hue="draft 长度", marker="o")
plt.title("Speculative Decoding 的理论加速趋势")
plt.xlabel("draft token acceptance rate")
plt.ylabel("approx speedup")
plt.show()

In [ ]:
# Continuous batching 的简化模拟：比较静态批处理与持续批处理的平均完成时间。
arrivals = np.array([0, 1, 2, 4, 5, 7, 9, 10, 12, 14])
service = np.array([6, 5, 7, 3, 4, 6, 5, 4, 3, 2])

static_finish = []
batch_size = 4
current_time = 0
for i in range(0, len(arrivals), batch_size):
    batch_arrival = arrivals[i : i + batch_size]
    batch_service = service[i : i + batch_size]
    start = max(current_time, batch_arrival.max())
    finish = start + batch_service.max()
    static_finish.extend([finish] * len(batch_arrival))
    current_time = finish

continuous_finish = []
current_time = 0
for a, s in zip(arrivals, service):
    start = max(current_time, a)
    finish = start + s
    continuous_finish.append(finish)
    current_time = finish

schedule_df = pd.DataFrame(
    {
        "请求": np.arange(len(arrivals)),
        "到达时间": arrivals,
        "静态批处理完成时间": static_finish,
        "持续调度完成时间": continuous_finish,
    }
)

schedule_df["静态响应时延"] = schedule_df["静态批处理完成时间"] - schedule_df["到达时间"]
schedule_df["持续响应时延"] = schedule_df["持续调度完成时间"] - schedule_df["到达时间"]
schedule_df

In [ ]:
latency_long = schedule_df.melt(
    id_vars="请求",
    value_vars=["静态响应时延", "持续响应时延"],
    var_name="策略",
    value_name="响应时延",
)
plt.figure(figsize=(11, 6))
sns.barplot(data=latency_long, x="请求", y="响应时延", hue="策略")
plt.title("静态批处理 vs 持续调度 的响应时延对比（玩具示意）")
plt.show()

## 4. 实战建议

- 长上下文首先检查 `KV Cache` 是否撑得住，而不是先纠结算力峰值
- 多请求服务场景里，`Continuous Batching` 往往比单点 kernel 优化更显著
- 想低成本部署，先考虑 `INT4 / INT8 + KV 优化 + Prefix Cache`
- 如果目标是极低时延，优先考虑 `Speculative Decoding + 小 draft model`
- 如果是高吞吐服务，`PagedAttention + 调度器 + 多 LoRA 复用` 会更关键

## 5. 主要资料

- vLLM 官方文档:
  https://docs.vllm.ai/en/v0.10.0/
- vLLM 官网:
  https://vllm.ai/
- FlashAttention（NeurIPS 2022）:
  https://arxiv.org/abs/2205.14135
- Fast Inference from Transformers via Speculative Decoding（2022-11-30）:
  https://arxiv.org/abs/2211.17192
- Accelerating Large Language Model Decoding with Speculative Sampling（2023-02-02）:
  https://arxiv.org/abs/2302.01318